# ML_LR_comparison — Colab Runner

動態學習率排程對比 + Grad-CAM 視覺化。支援兩個資料集：
- `tiny_imagenet` (200 類)
- `imagewoof` (10 類細粒度狗品種)

**前置：** Runtime → Change runtime type → 選 GPU (T4 / V100 / A100 皆支援，會自動套對應 profile)。

## 1. 確認 GPU

In [ ]:
!nvidia-smi

## 2. 掛載 Google Drive（用來持久化 logs / checkpoints）

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/ML_LR_comparison'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Drive output root:', DRIVE_ROOT)

## 3. Clone 本 repo + 安裝套件

In [ ]:
%cd /content
!rm -rf ML_LR_comparison
!git clone https://github.com/eric20041027/ML_LR_comparison.git
%cd /content/ML_LR_comparison
!pip install -q -r requirements.txt

## 4. 選擇資料集 + 共用參數

切換 `DATASET` 即可跑兩個資料集（或在同一 session 各跑一次）。每個資料集的輸出獨立在 Drive 一個子資料夾。

In [ ]:
import torch

# === 切換這裡：'tiny_imagenet' 或 'imagewoof' ===
DATASET = 'imagewoof'

DATA_DIR = '/content/data'              # Colab 暫存區，下載解壓後資料夾自動建在底下
OUTPUT_DIR = f'{DRIVE_ROOT}/experiments/{DATASET}'
EPOCHS = 20

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
PROFILE = 'a100' if 'A100' in gpu_name else 't4'
print(f'Dataset    : {DATASET}')
print(f'Detected   : {gpu_name}  ->  profile = {PROFILE}')
print(f'Output dir : {OUTPUT_DIR}')

## 5. 下載資料集（首次需要；已下載過會跳過）

In [ ]:
!python -m scripts.download_data --dataset {DATASET} --data-dir {DATA_DIR}

## 6. 跑 5 組實驗

輸出寫到 `{DRIVE_ROOT}/experiments/{DATASET}/`，斷線後可重連繼續觀察。

**預期時間（A100）：**
- Tiny-ImageNet：~25 分/組 × 5 = ~2 小時
- Imagewoof：~3 分/組 × 5 = ~15 分鐘（資料集小 10×）

In [ ]:
!python -m scripts.run_all \
    --dataset {DATASET} \
    --data-root {DATA_DIR} \
    --output-dir {OUTPUT_DIR} \
    --epochs {EPOCHS} \
    --profile {PROFILE}

## 7. 量化視覺化：LR 曲線 + Loss/Acc 曲線

In [ ]:
!python -m src.plot_lr --experiments-dir {OUTPUT_DIR} --out {OUTPUT_DIR}/lr_curves.png
!python -m src.plot_curves --experiments-dir {OUTPUT_DIR} --out {OUTPUT_DIR}/curves.png

from IPython.display import Image, display
display(Image(f'{OUTPUT_DIR}/lr_curves.png'))
display(Image(f'{OUTPUT_DIR}/curves.png'))

## 8. 質化視覺化：Grad-CAM 對比圖（5 排程 × 早/中/晚期）

`num_classes` 自動從各 run 的 `config.json` 讀取，所以兩個資料集共用同一指令。

In [ ]:
!python -m src.gradcam_viz \
    --experiments-dir {OUTPUT_DIR} \
    --data-root {DATA_DIR} \
    --out {OUTPUT_DIR}/grad_cam_grid.png

from IPython.display import Image, display
display(Image(f'{OUTPUT_DIR}/grad_cam_grid.png'))

## 9. (可選) 在 Colab 內看 TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir $OUTPUT_DIR